# Limpieza de los Datos de la Tabla bronce.pagos para Cargalos en la Capa Plata

- Proposito del script:  
    - Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
    - Limpiar y estandarizar, columna por columna.  
    - Exportar la nueva tabla como un archivo "limpio_pagos.parquet".  
- Procedimiento:  
    - Primera limpieza simple tipos de datos y logica basica.  
    - Eliminacion de duplicados luego de limpieza.  
    - Segunda limpieza mas compleja aplicando la logica de negocio.  
    
**Nota**: Esta tabla se limpia de esta manera porque, la informacion de las filas dependen de filas anteriores, en este caso, si existe pago_id duplicados, van a interferir constantemente con la limpieza.

# Estableciendo la Conexion

In [1]:
import pandas as pd
import numpy as np
from datetime import date 
from funciones import limpiar_texto, formato_canal_pago,recalcular_monto_interes_programado,recalcular_monto_mora_pagado, formato_estado_pago
from conexiones_y_rutas import obtener_engine,obtener_ruta_archivo
engine = obtener_engine()

df_pagos = pd.read_sql(
    "SELECT * FROM bronce.pagos",
    con=engine
)
df_pagos_tra = df_pagos.copy()

# Archivos de Ayuda

In [2]:
df_prestamos_tra = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_prestamos.parquet"))
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.11,...,10,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27
1,2,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,1473.65,...,7,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.44,...,35,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16
3,4,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,0.00,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-10-16
4,5,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06


# Resumen de las Columnas

- **pago_id**: Identificador unico de cada pago.  
- **prestamo_id**: Identificador del prestamo asociado a dicho pago.  
- **numero_cuota**: Numero de cuota del pago.  
- **fecha_vencimiento_cuota**: Fecha de vencimiento del pago.  
- **fecha_pago**: Fecha real de pago.  
- **monto_cuota_programada**: Monto de cuota programada.  
- **monto_capital_programado**: Monto capital programado.  
- **monto_interes_programado**: Monto de interes programado.
- **monto_pagado_total**: Monto real pagado por el cliente.
- **monto_capital_pagado**: Parte del monto total que se destina a reducir la deuda (monto real que se descuenta del total de la deuda).  
- **monto_interes_pagado**: Parte del monto que se destina a pagar intereses (monto que se paga por el interes NO SE DESCUENTA DE LA DEUDA).  
- **monto_mora_pagado**: Monto que se paga por la mora, solo se genera si existe mora en el pago (NO SE DESCUENTA DE LA DEUDA)
- **saldo_capital_despues_pago**: Saldo de la deuda despues de restar el monto_capital_pagado. 
- **dias_retraso**: Dias de retraso (diferencia entre fecha_vencimiento_cuota y fecha_pago).
- **estado_pago**: Situacion del pago (ejem: Puntual o Tardío).
- **canal_pago**: Canal donde se realizo el pago (ejem: Agencia o App Móvi).
- **referencia_pago**: Codigo de referencia del pago formato REF00000000(PAGO_ID) => 13 caracteres.

# Verificacion de la Calidad y Limpieza de los Datos

In [3]:
df_pagos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136944 entries, 0 to 136943
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   pago_id                     136944 non-null  int64  
 1   prestamo_id                 136944 non-null  int64  
 2   numero_cuota                136944 non-null  int64  
 3   fecha_vencimiento_cuota     136944 non-null  object 
 4   fecha_pago                  136944 non-null  object 
 5   monto_cuota_programada      136944 non-null  float64
 6   monto_capital_programado    136944 non-null  float64
 7   monto_interes_programado    136944 non-null  float64
 8   monto_pagado_total          136944 non-null  float64
 9   monto_capital_pagado        136944 non-null  float64
 10  monto_interes_pagado        136944 non-null  float64
 11  monto_mora_pagado           136944 non-null  float64
 12  saldo_capital_despues_pago  136944 non-null  float64
 13  dias_retraso  

In [4]:
df_pagos_tra.head()

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
0,1,1,1,2023-06-06,2023-06-06,393.63,238.91,154.72,393.63,238.91,154.72,0.0,8979.82,0,Puntual,App Móvil,REF0000000001
1,2,1,2,2023-07-06,2023-07-06,393.63,242.92,150.71,393.63,242.92,150.71,0.0,8736.90,0,Puntual,Débito Automático,REF0000000002
2,3,1,3,2023-08-05,2023-08-05,393.63,247.00,146.63,393.63,247.00,146.63,0.0,8489.90,0,Puntual,Agencia,REF0000000003
3,4,1,4,2023-09-04,2023-09-04,393.63,251.14,142.49,393.63,251.14,142.49,0.0,8238.76,0,Puntual,Agencia,REF0000000004
4,5,1,5,2023-10-04,2023-10-04,393.63,255.36,138.27,393.63,255.36,138.27,0.0,7983.40,0,Puntual,Débito Automático,REF0000000005


In [5]:
# Muestra registros duplicados
df_pagos_tra[df_pagos_tra.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
4139,4140,202,1,2023-09-27,2023-09-27,432.95,210.23,222.72,432.95,210.23,222.72,0.00,6153.31,0,Puntual,Agente Bancario,REF0000004140
4140,4140,202,1,2023-09-27,2023-09-27,432.95,210.23,222.72,432.95,210.23,222.72,0.00,6153.31,0,Puntual,Agente Bancario,REF0000004140
4141,4140,202,1,2023-09-27,2023-09-27,432.95,210.23,222.72,432.95,210.23,222.72,0.00,6153.31,0,Puntual,Agente Bancario,REF0000004140
4142,4140,202,1,2023-09-27,2023-09-27,432.95,210.23,222.72,432.95,210.23,222.72,0.00,6153.31,0,Puntual,Agente Bancario,REF0000004140
4143,4140,202,1,2023-09-27,2023-09-27,432.95,210.23,222.72,432.95,210.23,222.72,0.00,6153.31,0,Puntual,Agente Bancario,REF0000004140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136939,23569,1149,8,2022-11-21,2022-11-21,212.97,183.82,29.15,212.97,183.82,29.15,0.00,791.90,0,Puntual,Agencia,REF0000023569
136940,18162,885,31,2023-05-07,2023-07-14,-4150.31,612.88,3537.43,4291.42,612.88,3537.43,141.11,385991.52,68,Muy Tardío,Transferencia Interbancaria,REF0000018162
136941,46936,2269,7,2022-12-07,2022-12-07,1364.34,827.26,537.08,1364.34,827.26,537.08,0.00,31110.12,0,Puntual,App Móvil,REF0000046936
136942,57498,2772,7,2021-08-18,2021-08-18,231.31,128.22,103.09,231.31,128.22,103.09,0.00,6110.44,0,Puntual,App Móvil,REF0000057498


In [6]:
# Elimina registros duplicados y reinicia el indice
df_pagos_tra.drop_duplicates(inplace=True)
df_pagos_tra.reset_index(drop=True,inplace=True)

## pago_id

In [7]:
# Verifica si existen ids negativos o 0
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.pago_id <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [8]:
# Verifica si existen identificadores de pagos duplicados 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.pago_id.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## prestamo_id

In [9]:
# Verifica si todos los pagos tienen asociado un prestamo 
# Resultados Esperados: both: 134911, left_only: 0, right_only: 0
verificar_union = df_pagos_tra.merge(
    df_prestamos_tra,
    on='prestamo_id',
    how='left',
    indicator=True
)
verificar_union._merge.value_counts()

_merge
both          134911
left_only          0
right_only         0
Name: count, dtype: int64

## numero_cuota

**NOTA:** Falta revisar que el numero de cuota sea secuencial, es decir si tenemos como maximo el numero de cuotas pagadas 20, existan cuotas del 1 al 20.  
Tambien falta revisar que la cuota maxima registrada en pagos coincida con el numero de cuotas pagadas en prestamos.
Como existen ids duplicados, voy a realizar esta limpieza en:  
[Secuencialidad Numero Cuota](##Secuencialidad-numero_cuota)

In [10]:
# Verfica si existen valores negativos o 0 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.numero_cuota <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## fecha_vencimiento_cuota

In [11]:
# Fechas que generan error al transformar a date time
# Resultados Esperados: Tabla Vacia 
error_fecha_vencimiento = pd.to_datetime(df_pagos_tra.fecha_vencimiento_cuota,errors='coerce')
df_pagos_tra.fecha_vencimiento_cuota[error_fecha_vencimiento.isna()].head()

Series([], Name: fecha_vencimiento_cuota, dtype: object)

In [12]:
# Transforma las fechas a date time 
df_pagos_tra['fecha_vencimiento_cuota'] = pd.to_datetime(
    df_pagos_tra.fecha_vencimiento_cuota,
    errors='coerce',
    format='mixed',
    dayfirst=True
)

In [13]:
# Fechas que tenian error al inicio 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra.fecha_vencimiento_cuota[error_fecha_vencimiento.isna()].head()

Series([], Name: fecha_vencimiento_cuota, dtype: datetime64[ns])

## fecha_pago

In [14]:
# Fecha que generar error al transformar a date time
# Resultados Esperados: Tabla Vacia 
error_fecha_pago = pd.to_datetime(df_pagos_tra.fecha_pago,errors='coerce')
df_pagos_tra.fecha_pago[error_fecha_pago.isna()].head()

Series([], Name: fecha_pago, dtype: object)

In [15]:
# Transforma las fechas a date time 
df_pagos_tra['fecha_pago'] = pd.to_datetime(
    df_pagos_tra.fecha_pago,
    errors='coerce',
    format='mixed',
    dayfirst=True
)

In [16]:
# Fechas que tenian error al inicio 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra.fecha_pago[error_fecha_pago.isna()].head()

Series([], Name: fecha_pago, dtype: datetime64[ns])

In [17]:
# La limpieza de fecha_pago la voy a dejar para mas adelante, porque, me podria ayudar de dias_retraso y fecha_vencimiento_cuota, pero primero tendria que verificar que dias_retraso es correcto 
df_pagos_tra[df_pagos_tra.fecha_pago.isna()]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_cuota_programada

In [18]:
# Verifica si existen montos negativos o  0 
df_pagos_tra[df_pagos_tra.monto_cuota_programada <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
340,341,13,8,2023-11-29,2024-03-09,-1190.65,1105.23,85.42,1250.78,1105.23,85.42,60.13,4589.19,101,Muy Tardío,Transferencia Interbancaria,REF0000000341
519,520,21,15,2023-08-31,2023-09-02,-1658.05,1478.79,179.26,1658.05,1478.79,179.26,0.00,4698.87,2,Con Gracia,App Móvil,REF0000000520
836,837,40,9,2022-08-16,2022-11-09,-1880.76,470.25,1410.51,1960.69,470.25,1410.51,79.93,70055.48,85,Muy Tardío,App Móvil,REF0000000837
869,870,41,5,2024-08-17,2024-12-20,-70.65,36.21,34.44,75.07,36.21,34.44,4.42,1440.82,125,Muy Tardío,Agencia,REF0000000870
932,933,44,2,2021-12-18,2021-12-18,-1599.83,93.67,1506.16,1599.83,93.67,1506.16,0.00,164214.55,0,Puntual,Agencia,REF0000000933
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133848,133849,6435,44,2023-10-09,2023-10-09,-704.66,633.30,71.36,704.66,633.30,71.36,0.00,2673.14,0,Puntual,Agencia,REF0000133849
133860,133861,6436,8,2021-05-19,2021-05-19,-639.63,576.83,62.80,639.63,576.83,62.80,0.00,2430.38,0,Puntual,App Móvil,REF0000133861
134208,134209,6455,11,2024-10-21,2024-10-21,-262.52,248.25,14.27,262.52,248.25,14.27,0.00,255.28,0,Puntual,Cajero ATM,REF0000134209
134398,134399,6464,10,2024-05-26,2024-07-10,-767.45,713.12,54.33,784.72,713.12,54.33,17.27,1479.73,45,Muy Tardío,App Móvil,REF0000134399


In [19]:
# Aplica valor absoluto 
df_pagos_tra["monto_cuota_programada"] = df_pagos_tra.monto_cuota_programada.apply(abs)
df_pagos_tra[df_pagos_tra.monto_cuota_programada <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [20]:
# Calcula el promedio, porque, en caso exista una cuota con otro valor el resultado no va a ser igual entonces tendria que cambiar los valores 
group_cuota_programada = df_pagos_tra.groupby('prestamo_id').agg(
    cuota_pro = ('monto_cuota_programada','mean')
).reset_index()

# Redondea hasta las decenas porque, es el formato que se utiliza en prestamos
group_cuota_programada['cuota_pro'] = group_cuota_programada.cuota_pro.apply(lambda x: round(x,2))

# Left Join, prestamo_id 
merge_cuota_progra = group_cuota_programada.merge(
    right=df_prestamos_tra,
    on='prestamo_id',
    how='left'
)

# Verifica si las cuotas programadas en pagos son diferentes a las cuotas programadas en prestamos
merge_cuota_progra[['prestamo_id','cuota_pro','cuota_programada']][
    merge_cuota_progra.cuota_pro != merge_cuota_progra.cuota_programada
]

,prestamo_id,cuota_pro,cuota_programada


## monto_capital_programado

In [21]:
# Verifica si existen montos negativos o 0 
df_pagos_tra[df_pagos_tra.monto_capital_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [22]:
# Aplica valor absoluto 
df_pagos_tra["monto_capital_programado"] = df_pagos_tra.monto_capital_programado.apply(abs)
df_pagos_tra[df_pagos_tra.monto_capital_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_interes_programado

In [23]:
# Verifica si existen montos negativos o  0 
df_pagos_tra[df_pagos_tra.monto_interes_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [24]:
# Aplica valor absoluto 
df_pagos_tra["monto_interes_programado"] = df_pagos_tra.monto_interes_programado.apply(abs)
df_pagos_tra[df_pagos_tra.monto_interes_programado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_pagado_total

In [25]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_pagado_total <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
92,93,5,7,2020-11-15,2020-11-15,410.63,217.50,193.13,-410.63,217.50,193.13,0.00,6979.95,0,Puntual,Cajero ATM,REF0000000093
381,382,16,1,2022-03-12,2022-05-23,930.25,278.53,651.72,-963.74,278.53,651.72,33.49,25337.76,72,Muy Tardío,Agencia,REF0000000382
415,416,16,35,2024-12-26,2024-12-26,930.25,654.40,275.85,-930.25,654.40,275.85,0.00,10188.09,0,PUNTUAL,Débito Automático,REF0000000416
423,424,17,8,2023-12-08,2023-12-08,707.13,607.37,99.76,-707.13,607.37,99.76,0.00,4590.83,0,Puntual,Web Banca,REF0000000424
537,538,22,15,2021-04-05,2021-04-05,175.67,160.41,15.26,-175.67,160.41,15.26,0.00,503.72,0,Puntual,Web Banca,REF0000000538
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134573,134574,6474,22,2024-11-22,2024-11-22,731.74,511.05,220.69,-731.74,511.05,220.69,0.00,15978.61,0,Puntual,App Móvil,REF0000134574
134642,134643,6478,3,2023-02-26,2023-02-26,1126.92,324.23,802.69,-1126.92,324.23,802.69,0.00,102365.89,0,puntual,Cajero ATM,REF0000134643
134677,134678,6481,1,2022-08-25,2022-08-25,578.95,482.82,96.13,-578.95,482.82,96.13,0.00,2645.93,0,Puntual,Web Banca,REF0000134678
134848,134849,6491,17,2022-07-03,2022-07-03,257.50,215.27,42.23,-257.50,215.27,42.23,0.00,1649.72,0,Puntual,Agencia,REF0000134849


In [26]:
# Aplica valor absoluto 
df_pagos_tra["monto_pagado_total"] = df_pagos_tra.monto_pagado_total.apply(abs)
df_pagos_tra[df_pagos_tra.monto_pagado_total <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_capital_pagado

In [27]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_capital_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [28]:
# Aplicamos valor absoluto 
df_pagos_tra["monto_capital_pagado"] = df_pagos_tra.monto_capital_pagado.apply(abs)
df_pagos_tra[df_pagos_tra.monto_capital_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_interes_pagado

In [29]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_interes_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [30]:
# Aplica valor absoluto 
df_pagos_tra["monto_interes_pagado"] = df_pagos_tra.monto_interes_pagado.apply(abs)
df_pagos_tra[df_pagos_tra.monto_interes_pagado <= 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_mora_pagado

In [31]:
# Verifica si existen montos negativos o  0 
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.monto_mora_pagado < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [32]:
# Aplica valor absoluto 
df_pagos_tra["monto_mora_pagado"] = df_pagos_tra.monto_mora_pagado.apply(abs)
df_pagos_tra[df_pagos_tra.monto_mora_pagado < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## saldo_capital_despues_pago

In [33]:
# Verifica si existen saldos negativos 0
# Resultados Esperados: Tabla Vacia
df_pagos_tra[df_pagos_tra.saldo_capital_despues_pago < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [34]:
# Aplica valor absoluto 
df_pagos_tra["saldo_capital_despues_pago"] = df_pagos_tra.saldo_capital_despues_pago.apply(abs)
df_pagos_tra[df_pagos_tra.saldo_capital_despues_pago < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## dias_retraso

In [35]:
# Verifica si existen saldos negativos 0
df_pagos_tra[df_pagos_tra.dias_retraso < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [36]:
# Aplica valor absoluto 
df_pagos_tra["dias_retraso"] = df_pagos_tra.dias_retraso.apply(abs)
df_pagos_tra[df_pagos_tra.dias_retraso < 0]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## estado_pago

In [37]:
# Verifica el formato del texto 
# Resultados Esperados: 'Puntual', 'Muy Tardío', 'Con Gracia', 'Tardío'
df_pagos_tra.estado_pago.unique()

array(['Puntual', 'Muy Tardío', 'Con Gracia', 'PUNTUAL', 'Puntual ',
       'Tardío', 'puntual'], dtype=object)

In [38]:
# Arregla el formato del estado del pago
df_pagos_tra["estado_pago"] = df_pagos_tra.estado_pago.apply(limpiar_texto)
df_pagos_tra.estado_pago.unique()

array(['Puntual', 'Muy Tardío', 'Con Gracia', 'Tardío'], dtype=object)

## canal_pago

In [39]:
# Verifica el formato del texto 
# Resultados Esperados: 'APP Móvil', 'Débito Automático', 'Agencia', 'Transferencia Interbancaria', 'Cajero ATM',      'Web Bancaria', 'Agente Bancario', n/a'

df_pagos_tra.canal_pago.unique()

array(['App Móvil', 'Débito Automático', 'Agencia',
       'Transferencia Interbancaria', 'Cajero ATM', 'App Móvil ',
       'Web Banca', 'Agente Bancario', 'agencia', 'app movil', None,
       'Agencia ', 'APP MOVIL', 'App Movil', 'AGENCIA',
       'Débito automático', 'DEBITO AUTOMATICO', 'debito automatico'],
      dtype=object)

In [40]:
# Arregla el formato del estado del pago
df_pagos_tra["canal_pago"] = df_pagos_tra.canal_pago.apply(formato_canal_pago)
df_pagos_tra.canal_pago.unique()

array(['APP Móvil', 'Débito Automático', 'Agencia',
       'Transferencia Interbancaria', 'Cajero ATM', 'Web Bancaria',
       'Agente Bancario', 'n/a'], dtype=object)

## referencia_pago

In [41]:
# Verifica si existen valores con espacios vacios innecesarios o que no tengas 13 caracteres 
# Resultados Esperados: Tabla Vacia
df_pagos_tra.referencia_pago[
    (df_pagos_tra.referencia_pago
        != df_pagos_tra.referencia_pago.str.strip().str.upper())
    |
    (df_pagos_tra.referencia_pago.apply(len)!= 13)
]

Series([], Name: referencia_pago, dtype: object)

In [42]:
# Verifica el formato de la referencia del pago REF00000000(PAGO_ID) => 13 caracteres 
# Resultados Esperados: Tabla Vacia
canal_pag_gene = (
    "REF"
    + df_pagos_tra["pago_id"]
        .astype(str)
        .str.zfill(10)
)

df_pagos_tra[df_pagos_tra.referencia_pago != canal_pag_gene]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


# Limpieza duplicados luego de primera limpieza

In [43]:
df_pagos_tra[df_pagos_tra.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [44]:
df_pagos_tra.drop_duplicates(inplace=True)
df_pagos_tra.reset_index(inplace=True,drop=True)

## pago_id V2

In [45]:
df_pagos_tra[df_pagos_tra.pago_id.duplicated(keep=False)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


# Segunda verificacion de la calidad de columnas y tablas

## Secuencialidad numero_cuota V2

In [46]:
# Verifica que existan registros secuenciales, es decir si tenemos como maximo el numero de cuotas pagadas 20, existan cuotas del 1 al 20 
# Es similar a la funcion de ventana ROW_NUMBER() en SQL server. 

# Ordena por prestamo_id y numero de cuota, no se usa fecha_pago, porque no esta limpia
verificar_secuencia = df_pagos_tra.copy()
verificar_secuencia = verificar_secuencia.sort_values(by=['prestamo_id','numero_cuota'],ascending=True)
verificar_secuencia['row_number'] = (
    verificar_secuencia
    .groupby(by='prestamo_id')
    .cumcount() + 1
)

# Verifica si existen registros donde el numero de cuota no coincida con el numero de registro de ese prestamo
# Resultados Esperados: Tabla Vacia
verificar_secuencia[['pago_id','prestamo_id','numero_cuota','row_number']][
    verificar_secuencia.numero_cuota != verificar_secuencia.row_number
]

,pago_id,prestamo_id,numero_cuota,row_number


In [47]:
# Selecciona el maximo numero de cuota para cada prestamo_id 
group_pagos = df_pagos_tra.groupby('prestamo_id').agg(
    max_num_cuota = ('numero_cuota','max')
).reset_index()

# Left Join, tabla agrupada de pagos y prestamos 
merge_pagos_prestamos = group_pagos.merge(
    right= df_prestamos_tra,
    on='prestamo_id',
    how='left'
)

merge_pagos_prestamos.head()

,prestamo_id,max_num_cuota,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
0,1,20,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,...,10,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27
1,2,47,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,...,7,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23
2,3,13,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,...,35,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16
3,4,6,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-10-16
4,5,30,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06


In [48]:
# Verifica que el numero maximo de cuota por prestamo_id sea igual al numero de cuotas pagadas en la tabla limpio_prestamos
# Resultados Esperados: Tabla Vacia 
merge_pagos_prestamos[merge_pagos_prestamos.max_num_cuota != merge_pagos_prestamos.numero_cuotas_pagadas]

,prestamo_id,max_num_cuota,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## Verificar fecha_vencimiento_cuota V2

**Nota**: Como vemos que fecha vencimiento para la cuota 1 es igual a la fecha fecha_primer_pago_programado de prestamos (solo existen 2 registros que no coinciden de mas de 6k prestamos) vamos a tomar esta fecha como la correcta y realizar los demás pagos siguiendo esta fecha.  
- El problema se genera porque, en algunos puntos, se rompe con la logica de cronogramas de pago (fecha_anterior + 30 dias), por ello se recalcula estas fechas.
- Teniendo en cuenta el interes pagado y el interes programado, puedo entender que realmente si pasaron 30 dias porque caso contrario el interes cambiaria. 
- En un entorno real, tendria que buscar informacion adicional y registros para poder validar si la informacion es correcta, para fines practicos de este proyecto, voy a cambiar los datos.

In [49]:
# fecha de vencimiento para la cuota numero 1 
df_group_fecha_min = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota']][df_pagos_tra.numero_cuota == 1].copy()

df_fecha_venci = df_prestamos_tra[['prestamo_id','fecha_primer_pago_programado','fecha_ultimo_pago_real']].copy()

# LEFT JOIN, por prestamo_id
df_merge_fecha_min_fecha_venci = df_group_fecha_min.merge(
    right=df_fecha_venci,
    on='prestamo_id',
    how='left'
)

# Verifica que la fecha_vencimiento sea igual a la fecha_primer_pago_programado establecido en prestamos
df_merge_fecha_min_fecha_venci[
    df_merge_fecha_min_fecha_venci.fecha_vencimiento_cuota
        != df_merge_fecha_min_fecha_venci.fecha_primer_pago_programado
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,fecha_primer_pago_programado,fecha_ultimo_pago_real
1853,38112,1854,2024-10-12,2025-01-08,2024-12-11
4609,96214,4610,2020-04-11,2021-01-02,2020-12-07


**Nota**: La razon principal de usar fecha_primer_pago_programado de prestamos, es porque, en prestamos coincide a l 100% fecha_otorgamiento + 30 dias = fecha_primer_pago_programado.

In [50]:
# Reemplaza la fecha de vencimiento de la cuota de prestamos utilizando, fecha_primer_pago_programado 
# Como solo son 2 registros voy a realizar estos cambios de manera manual 
df_pagos_tra.loc[df_pagos_tra.pago_id == 38112,"fecha_vencimiento_cuota"] = pd.to_datetime("2025-01-08")

df_pagos_tra.loc[df_pagos_tra.pago_id == 96214,"fecha_vencimiento_cuota"] = pd.to_datetime("2021-01-02")


In [51]:
pagos_ordenados = (
    df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota']]
    .sort_values(by=['prestamo_id','numero_cuota'])
    .copy()
)
# Selecciona el valor del registro anterior, similar a LAG() en SQL
pagos_ordenados['fecha_anterior'] = (
    pagos_ordenados
    .groupby('prestamo_id')['fecha_vencimiento_cuota']
    .shift(1)
)
# Muestra las fechas que no siguen la logica de sumar 30 dias por cada nuevo pago
# Resultados Esperados: Tabla Vacia  
# fecha_vencimiento_cuota_N = fecha_vencimiento_cuota_N-1 + 30
pagos_ordenados[
    (pagos_ordenados.fecha_vencimiento_cuota 
        != pagos_ordenados.fecha_anterior+pd.DateOffset(days=30))
    & 
    (pagos_ordenados.fecha_anterior.notna())
]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_anterior
108,109,5,23,2022-10-03,2022-02-08
109,110,5,24,2022-04-09,2022-10-03
129,130,6,14,2021-01-10,2021-09-01
130,131,6,15,2021-10-31,2021-01-10
392,393,16,12,2023-05-02,2023-01-06
...,...,...,...,...,...
133914,133915,6439,10,2023-11-06,2023-10-31
134053,134054,6446,10,2022-12-06,2022-05-13
134054,134055,6446,11,2022-07-12,2022-12-06
134602,134603,6475,28,2024-09-01,2023-12-10


In [52]:
# Columnas a utilizar
df_fecha_venci = df_pagos_tra[['pago_id','prestamo_id','numero_cuota']].copy()
df_presta_revi = df_prestamos_tra[['prestamo_id','fecha_primer_pago_programado']].copy()

# Left join, prestamo_id 
df_merge_fecha_vencimie = df_fecha_venci.merge(
    right=df_presta_revi,
    on='prestamo_id',
    how='left'
)


# Recalcula la fecha de vencimiento añadiendo 30 dias por cada nuevo pago 
# La razon porque, se trabaja con 30 dias y no añadiendo 1 mes, es por la logica observada en los demas prestamos
# fecha_venci_recal = fecha_primer_pago_programado + (numero_cuota-1)*30
# La razon porque es numero_cuota - 1, es porque la fecha de primer pago ya tiene + 30 dias
df_merge_fecha_vencimie['fecha_venci_recal'] = df_merge_fecha_vencimie.apply(
    axis= 1,
    func= lambda x : 
        x['fecha_primer_pago_programado'] + pd.DateOffset(days=30*(x['numero_cuota']-1))
)

# reemplazamos por los valores recalculados 
df_pagos_tra['fecha_vencimiento_cuota'] = df_merge_fecha_vencimie.fecha_venci_recal

In [53]:
pagos_ordenados = (
    df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota']]
    .sort_values(by=['prestamo_id','numero_cuota'])
    .copy()
)
# Selecciona el valor del registro anterior, similar a LAG() en SQL
pagos_ordenados['fecha_anterior'] = (
    pagos_ordenados
    .groupby('prestamo_id')['fecha_vencimiento_cuota']
    .shift(1)
)
# Muestra las fechas que no siguen la logica de sumar 30 dias por cada nuevo pago 
# Resultados Esperados: Tabla Vacia 
pagos_ordenados[
    (pagos_ordenados.fecha_vencimiento_cuota 
        != pagos_ordenados.fecha_anterior+pd.DateOffset(days=30))
    & 
    (pagos_ordenados.fecha_anterior.notna())
]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_anterior


In [54]:
# Muestra las fechas de vencimiento futuras 
# Resultados Esperados: Tabla Vacia 
df_pagos_tra[df_pagos_tra.fecha_vencimiento_cuota.dt.date >= date.today()]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## fecha_pago y dias retraso

In [55]:
# Columnas a utilizar 
verificar_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

verificar_dias_retraso['dias_retraso_recal'] = verificar_dias_retraso.apply(
    axis=1,
    func= lambda x: 
        (x['fecha_pago'] - x['fecha_vencimiento_cuota']).days
        if x['fecha_pago'] > x['fecha_vencimiento_cuota']
        else 0
)

# Muestra los registros donde el recalculo de los dias de retraso es diferente a los dias de retraso original 
# Resultados Esperados: Tabla Vacia 
verificar_dias_retraso[
    verificar_dias_retraso.dias_retraso != verificar_dias_retraso.dias_retraso_recal
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,monto_mora_pagado,fecha_pago,dias_retraso,dias_retraso_recal
108,109,5,2022-03-10,0.00,2022-10-04,1,208
129,130,6,2021-10-01,0.00,2021-01-13,3,0
392,393,16,2023-02-05,0.00,2023-05-01,0,85
1187,1188,51,2024-11-12,0.00,2024-12-14,3,32
1254,1255,54,2022-03-12,52.43,2022-06-14,0,94
...,...,...,...,...,...,...,...
133912,133913,6439,2024-05-28,0.00,2023-11-02,2,0
133917,133918,6439,2024-10-25,32.06,2024-05-14,100,0
133919,133920,6439,2024-12-24,0.00,2024-04-06,2,0
134053,134054,6446,2022-06-12,0.00,2022-12-07,1,178


## monto_cuota_programada V2

**Nota**: Ya se verifico que esta cuota sea la misma cuota establecida en la tabla_bronce.prestamo  
[cuota_programada](#!monto_cuota_programada)

In [56]:
# Verifica que se cumpla cuota = interes + amortizacion 
df_pagos_tra[df_pagos_tra.monto_cuota_programada 
            != round(df_pagos_tra.monto_capital_programado + df_pagos_tra.monto_interes_programado,2)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_capital_programado y monto_interes_programado 

In [57]:
# Columnas a utilizar 
df_prestamos_veri_montos = df_prestamos_tra[['prestamo_id','tasa_interes_efectiva_anual','numero_cuotas_total','monto_original']].copy()
# Calcula el el interes mensual para cada prestamo
df_prestamos_veri_montos['tasa_efectiva_mensual'] = df_prestamos_veri_montos.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/12)-1)*100
)
#--------------------------------------------------------------------------------
# Verifica que monto_capital_programado y monto_interes_programado sean correctos
#--------------------------------------------------------------------------------
df_verificar_interes_pro = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','monto_cuota_programada','monto_capital_programado','monto_interes_programado']].copy()

# LEFT JOIN, prestamo_id 
df_merge_veri_inte_pro = df_verificar_interes_pro.merge(
    right=df_prestamos_veri_montos,
    on='prestamo_id',
    how='left'
)

df_merge_veri_inte_pro.sort_values(['prestamo_id','numero_cuota'],inplace=True)

In [58]:
# Crea nuevas columnas para validar, considerando que monto capital programado es correcto
df_merge_veri_inte_pro['acumulado_capital'] = df_merge_veri_inte_pro.groupby(['prestamo_id'])['monto_capital_programado'].cumsum()

# Columnas para verificar 
df_merge_veri_inte_pro['saldo'] = df_merge_veri_inte_pro.apply(
    axis = 1,
    func = lambda x: round(x['monto_original'] - x['acumulado_capital'],2)
)

df_merge_veri_inte_pro['saldo_para_interes'] = df_merge_veri_inte_pro.groupby(['prestamo_id'])['saldo'].shift(1)

df_merge_veri_inte_pro['interes_recal'] = df_merge_veri_inte_pro.apply(
    axis = 1,
    func = lambda x: recalcular_monto_interes_programado(x['monto_original'] ,x['numero_cuota'],x['tasa_efectiva_mensual'] ,x['numero_cuotas_total'], x['monto_cuota_programada'] , x['saldo_para_interes'] )
)
# Para esta verificacion vamos a considerar un margen de +- 0.04 por errores en el redondeo
# Regresa todos los valores que no se encuentre en el rango de +- 0.04
df_merge_veri_inte_pro[~(df_merge_veri_inte_pro.monto_interes_programado.between(
    left = df_merge_veri_inte_pro.interes_recal - 0.04,
    right= df_merge_veri_inte_pro.interes_recal + 0.04,
    inclusive='both')
    )
]

,pago_id,prestamo_id,numero_cuota,monto_cuota_programada,monto_capital_programado,monto_interes_programado,tasa_interes_efectiva_anual,numero_cuotas_total,monto_original,tasa_efectiva_mensual,acumulado_capital,saldo,saldo_para_interes,interes_recal
0,1,1,1,393.63,238.91,154.72,22.1071,30,9218.73,1.678331,238.91,8979.82,NaN,NaN
20,21,2,1,225.92,86.37,139.55,23.8234,54,7767.38,1.796667,86.37,7681.01,NaN,NaN
67,68,3,1,495.20,119.84,375.36,42.5761,48,12512.04,3.000001,119.84,12392.20,NaN,NaN
80,81,4,1,667.82,542.35,125.47,51.6186,6,3555.11,3.529164,542.35,3012.76,NaN,NaN
86,87,5,1,410.63,185.55,225.08,37.4040,30,8388.15,2.683331,185.55,8202.60,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134832,134833,6491,1,257.50,150.46,107.04,30.8223,24,4727.66,2.264166,150.46,4577.20,NaN,NaN
134856,134857,6492,1,397.28,342.26,55.02,21.9871,9,3294.34,1.670000,342.26,2952.08,NaN,NaN
134865,134866,6493,1,3405.04,162.21,3242.83,10.6796,360,381883.83,0.849163,162.21,381721.62,NaN,NaN
134892,134893,6494,1,853.34,798.18,55.16,30.6433,3,2448.88,2.252498,798.18,1650.70,NaN,NaN


## saldo_capital_despues_pago V2

In [59]:
# Verifica que el saldo calculado sea igual al saldo_capital_despues_pago
# para esta verificacion aprovechamos el calculo anterior de saldo
df_verificar_saldo_capital_despues_pago = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','saldo_capital_despues_pago']].sort_values(['prestamo_id','numero_cuota']).copy()

df_verificar_saldo_capital_despues_pago[df_verificar_saldo_capital_despues_pago.saldo_capital_despues_pago != df_merge_veri_inte_pro.saldo]

,pago_id,prestamo_id,numero_cuota,saldo_capital_despues_pago


In [60]:
#--------------------------------------------------------------------------------------------------------------
# Verifca si el saldo capital_vigente de prestamos es igual al ultimo saldo_capital_despues_pago para cada pago 
#--------------------------------------------------------------------------------------------------------------
# Obtiene saldo capital vigente despues de pagar la ultima cuota de cada prestamo
idx_cuota_max = df_verificar_saldo_capital_despues_pago.groupby('prestamo_id').agg(
    idx_num_cuota_max = ('numero_cuota','idxmax')
).reset_index()
df_verificar_saldo_capital_despues_pago = df_verificar_saldo_capital_despues_pago.loc[idx_cuota_max.idx_num_cuota_max]

df_prestamos_veri_saldo_capi = df_prestamos_tra[['prestamo_id','saldo_capital_vigente']].copy() 

# LEFT JOIN, prestamo_id 
df_merge_veri_saldo_capital = df_verificar_saldo_capital_despues_pago.merge(
    right=df_prestamos_veri_saldo_capi,
    on='prestamo_id',
    how='left'
)

# Muestra los registros donde no coinciden los valores 
# El cambio se va a realizar en 09_validacion_pagos_prestamos 
df_merge_veri_saldo_capital[df_merge_veri_saldo_capital.saldo_capital_despues_pago != df_merge_veri_saldo_capital.saldo_capital_vigente]

,pago_id,prestamo_id,numero_cuota,saldo_capital_despues_pago,saldo_capital_vigente
0,20,1,20,3596.20,3596.11
1,67,2,47,1473.62,1473.65
2,80,3,13,10640.43,10640.44
5,169,6,53,188558.01,188558.10
9,281,10,43,18910.91,32134.06
...,...,...,...,...,...
6483,134726,6484,10,207381.39,207381.43
6484,134739,6485,13,738.75,738.70
6489,134832,6490,30,0.00,2395.83
6492,134892,6493,27,376984.64,376984.52


## monto_capital_pagado V2

In [61]:
# verifica si el monto pagado no coincide con el monto programado
df_pagos_tra[df_pagos_tra.monto_capital_pagado != df_pagos_tra.monto_capital_programado]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_interes_pagado V2

In [62]:
# verifica si el monto pagado no coincide con el monto programado
df_pagos_tra[df_pagos_tra.monto_interes_pagado != df_pagos_tra.monto_interes_programado]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## monto_pagado_total V2

**Nota**: Como vemos que capital e interes pagado coinciden con los programados los podemos considerar columnas ancla, entonces ahora lo que tendríamos que revisar es el monto_mora_pagado o monto_pagado_total.

In [63]:
# Muestra las columnas donde no coincida el monto pagado con el monto capital, el monto intereses y el monto de mora
df_pagos_total_incorrecto = df_pagos_tra[
    df_pagos_tra.monto_pagado_total.round(2) 
        != (df_pagos_tra.monto_capital_pagado + df_pagos_tra.monto_interes_pagado + df_pagos_tra.monto_mora_pagado).round(2)
]
df_pagos_total_incorrecto

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
2494,2495,115,24,2023-09-12,2023-11-18,862.03,839.84,22.19,890.91,839.84,22.19,14440.0,0.00,67,Muy Tardío,Agencia,REF0000002495
4830,4831,238,2,2020-09-02,2020-12-03,362.54,262.56,99.98,379.22,262.56,99.98,8340.0,3096.24,92,Muy Tardío,Agencia,REF0000004831
5356,5357,264,24,2023-01-10,2023-04-12,1211.70,337.72,873.98,1267.44,337.72,873.98,27870.0,123048.00,92,Muy Tardío,Agencia,REF0000005357
7567,7568,366,2,2023-04-27,2023-09-15,69.47,16.44,53.03,74.37,16.44,53.03,2450.0,2127.75,141,Muy Tardío,Agencia,REF0000007568
8456,8457,413,14,2022-03-29,2022-06-23,938.71,458.33,480.38,979.07,458.33,480.38,20180.0,62958.22,86,Muy Tardío,Agente Bancario,REF0000008457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128681,128682,6171,33,2024-07-13,2024-08-15,5306.08,1807.81,3498.27,5393.63,1807.81,3498.27,43775.0,379821.53,33,Muy Tardío,Web Bancaria,REF0000128682
129713,129714,6224,9,2021-06-09,2021-09-15,1082.74,495.43,587.31,1135.79,495.43,587.31,26525.0,33766.79,98,Muy Tardío,Agente Bancario,REF0000129714
130053,130054,6239,5,2020-06-20,2020-10-11,172.62,110.56,62.06,182.37,110.56,62.06,4875.0,2644.43,113,Muy Tardío,Cajero ATM,REF0000130054
130128,130129,6243,11,2023-07-21,2023-11-07,288.49,191.87,96.62,304.21,191.87,96.62,7860.0,4498.21,109,Muy Tardío,APP Móvil,REF0000130129


## Revision fecha_pago, dias_retraso y monto_mora_pagado

In [64]:
# Verifica si el monto mora es correcto para los registros con errores 
# De prestamos 
df_prestamos_veri_mora = df_prestamos_tra[['prestamo_id','tasa_interes_efectiva_anual','numero_cuotas_total','monto_original']].copy()

# Calcula el interes diario para cada prestamo 
df_prestamos_veri_mora['tasa_efectiva_diaria'] = df_prestamos_veri_mora.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/360)-1)*100
)
#--------------------------------------------------------
# Recalcula monto_mora_pagado utilizando los dias_retraso
#--------------------------------------------------------
veri_monto_mora = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_vencimiento_cuota','monto_capital_programado','monto_pagado_total','monto_capital_pagado','monto_interes_pagado','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

df_merge_monto_mora = df_prestamos_veri_mora.merge(
    right=veri_monto_mora,
    how='right',
    on='prestamo_id'
)
df_merge_monto_mora.sort_values(['prestamo_id','numero_cuota'],inplace=True)
# Columnas a utilizar 
df_merge_monto_mora['acumulado_capital'] = df_merge_monto_mora.groupby(['prestamo_id'])['monto_capital_programado'].cumsum()
df_merge_monto_mora['saldo'] = df_merge_monto_mora.apply(
    axis = 1,
    func = lambda x: round(x['monto_original'] - x['acumulado_capital'],2)
) 

# Selecciona el saldo para el interes usando los registros anteriores, similar a LAG(), en sql
df_merge_monto_mora['saldo_para_interes'] = df_merge_monto_mora.groupby(['prestamo_id'])['saldo'].shift(1)

df_merge_monto_mora['monto_mora_recal'] = df_merge_monto_mora.apply(
    axis = 1,
    func= lambda x:  recalcular_monto_mora_pagado(x['numero_cuota'],x['saldo_para_interes'],x['monto_original'], x['tasa_efectiva_diaria'],x['dias_retraso'])
)

# Muestra los registros donde no coindia el monto_mora_pagado con monto_mora_recal 
# Se considera un margen de +-0.4 por el redondeo
# Resultados Esperados: Tabla Vacia 
df_pagos_mora_inco = df_merge_monto_mora[['pago_id','prestamo_id','tasa_efectiva_diaria','monto_pagado_total','monto_capital_pagado','monto_interes_pagado','saldo_para_interes','dias_retraso','monto_mora_pagado','monto_mora_recal']][
    ~(df_merge_monto_mora.monto_mora_pagado.between(
    left = df_merge_monto_mora.monto_mora_recal - 0.04,
    right= df_merge_monto_mora.monto_mora_recal + 0.04,
    inclusive='both'))
    ]
df_pagos_mora_inco

,pago_id,prestamo_id,tasa_efectiva_diaria,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,saldo_para_interes,dias_retraso,monto_mora_pagado,monto_mora_recal
0,1,1,0.055495,393.63,238.91,154.72,NaN,0,0.00,NaN
9,10,1,0.055495,410.16,277.52,116.11,6918.39,84,16.53,330.05
12,13,1,0.055495,393.63,291.73,101.90,6071.79,3,0.00,10.11
18,19,1,0.055495,393.63,322.36,71.27,4246.33,5,0.00,11.80
20,21,2,0.059375,225.92,86.37,139.55,NaN,0,0.00,NaN
...,...,...,...,...,...,...,...,...,...,...
134894,134895,6494,0.074278,897.29,834.54,18.80,834.54,103,43.95,66.33
134895,134896,6495,0.029980,753.70,56.54,697.16,NaN,0,0.00,NaN
134899,134900,6495,0.029980,753.70,58.62,695.08,76946.61,6,0.00,138.52
134901,134902,6495,0.029980,753.70,59.68,694.02,76828.84,7,0.00,161.38


In [65]:
# Verifica los errores en dias retraso 
verificar_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','fecha_vencimiento_cuota','monto_mora_pagado','fecha_pago','dias_retraso']].copy()

verificar_dias_retraso['dias_retraso_recal'] = verificar_dias_retraso.apply(
    axis=1,
    func= lambda x: 
        (x['fecha_pago'] - x['fecha_vencimiento_cuota']).days
        if x['fecha_pago'] > x['fecha_vencimiento_cuota']
        else 0
)

verificar_dias_retraso[
    verificar_dias_retraso.dias_retraso != verificar_dias_retraso.dias_retraso_recal
]

,pago_id,prestamo_id,fecha_vencimiento_cuota,monto_mora_pagado,fecha_pago,dias_retraso,dias_retraso_recal
108,109,5,2022-03-10,0.00,2022-10-04,1,208
129,130,6,2021-10-01,0.00,2021-01-13,3,0
392,393,16,2023-02-05,0.00,2023-05-01,0,85
1187,1188,51,2024-11-12,0.00,2024-12-14,3,32
1254,1255,54,2022-03-12,52.43,2022-06-14,0,94
...,...,...,...,...,...,...,...
133912,133913,6439,2024-05-28,0.00,2023-11-02,2,0
133917,133918,6439,2024-10-25,32.06,2024-05-14,100,0
133919,133920,6439,2024-12-24,0.00,2024-04-06,2,0
134053,134054,6446,2022-06-12,0.00,2022-12-07,1,178


In [66]:
# Verifica si en los registros con monto_mora incorrectos se encuentras los pagos_totales_incorrectos 
df_pagos_mora_inco.loc[df_pagos_mora_inco.pago_id.isin(df_pagos_total_incorrecto.pago_id)]

,pago_id,prestamo_id,tasa_efectiva_diaria,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,saldo_para_interes,dias_retraso,monto_mora_pagado,monto_mora_recal
2494,2495,115,0.086707,890.91,839.84,22.19,839.84,67,14440.0,50.21
4830,4831,238,0.097822,379.22,262.56,99.98,3358.80,92,8340.0,316.14
5356,5357,264,0.023531,1267.44,337.72,873.98,123385.72,92,27870.0,2699.88
7567,7568,366,0.081475,74.37,16.44,53.03,2144.19,141,2450.0,260.92
8456,8457,413,0.025158,979.07,458.33,480.38,63416.55,86,20180.0,1386.84
...,...,...,...,...,...,...,...,...,...,...
128681,128682,6171,0.030421,5393.63,1807.81,3498.27,381629.34,33,43775.0,3849.86
129713,129714,6224,0.056671,1135.79,495.43,587.31,34262.22,98,26525.0,1956.09
130053,130054,6239,0.074278,182.37,110.56,62.06,2754.99,113,4875.0,241.13
130128,130129,6243,0.067992,304.21,191.87,96.62,4690.08,109,7860.0,360.67


**Nota**: Revisando veo que existen 89 registros donde no coincide el monto total, por otro lado, al recalcular la mora, tomando como valor correcto los dias de retraso me generan 42.6k registros incorrectos, voy a considerar monto_pagado_total como correcto y en base a ello voy a recalcular monto_mora_pagado, dias retraso y fecha_de_pago  

## monto_mora_pagado V2

In [67]:
# monto_mora_pagado = monto_pagado_total -  monto_capital_pagado - monto_interes_pagado
df_pagos_tra['monto_mora_pagado'] = df_pagos_tra.apply(
    axis = 1,
    func = lambda x: abs(round(x['monto_pagado_total'] - x['monto_capital_pagado'] - x['monto_interes_pagado'],2))
)

df_pagos_tra[df_pagos_tra.monto_pagado_total.round(2) != (df_pagos_tra.monto_capital_pagado + df_pagos_tra.monto_interes_pagado + df_pagos_tra.monto_mora_pagado).round(2)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


## dias_retraso V2

**Formula para calcular el periodo (dias)**:  
 ln(1+ I/VA) / ln(1 + I)

In [68]:
# calculamos los días de retraso en funcion de monto_mora_pagado 
df_verificar_retraso = df_pagos_tra[['pago_id','prestamo_id','monto_capital_pagado','monto_mora_pagado','saldo_capital_despues_pago','dias_retraso']].copy()
df_merge_veri_dias_retraso = df_verificar_retraso.merge(
    right=df_prestamos_veri_mora,
    on='prestamo_id',
    how='left'
)

df_merge_veri_dias_retraso['saldo_para_interes'] = df_merge_veri_dias_retraso.apply(
    axis = 1,
    func = lambda x: 
    round(
        x['saldo_capital_despues_pago'] + x['monto_capital_pagado'],
        2
    )
)
# Recalcula los dias de retraso, en este caso realizo un redondeo sin decimales
df_merge_veri_dias_retraso['recal_dias_retraso'] = df_merge_veri_dias_retraso.apply(
    axis = 1,
    func = lambda x: 
    round(
        np.log(1+(x['monto_mora_pagado']/x['saldo_para_interes'])) / np.log(1 + x['tasa_efectiva_diaria']/100)
    ) 
)
df_merge_veri_dias_retraso.head()

,pago_id,prestamo_id,monto_capital_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,tasa_interes_efectiva_anual,numero_cuotas_total,monto_original,tasa_efectiva_diaria,saldo_para_interes,recal_dias_retraso
0,1,1,238.91,0.0,8979.82,0,22.1071,30,9218.73,0.055495,9218.73,0
1,2,1,242.92,0.0,8736.90,0,22.1071,30,9218.73,0.055495,8979.82,0
2,3,1,247.00,0.0,8489.90,0,22.1071,30,9218.73,0.055495,8736.90,0
3,4,1,251.14,0.0,8238.76,0,22.1071,30,9218.73,0.055495,8489.90,0
4,5,1,255.36,0.0,7983.40,0,22.1071,30,9218.73,0.055495,8238.76,0


In [69]:
# reemplazo los valores en el df original 
df_pagos_tra['dias_retraso'] = df_merge_veri_dias_retraso['recal_dias_retraso']

dias_retra_sin_pago = df_pagos_tra[(df_pagos_tra.dias_retraso != 0) & (df_pagos_tra.monto_mora_pagado == 0)].shape[0]
sin_dias_retra_con_pago = df_pagos_tra[(df_pagos_tra.dias_retraso == 0) & (df_pagos_tra.monto_mora_pagado != 0)].shape[0]
print(f"Registros con dias de retraso y sin pago por mora: {dias_retra_sin_pago}")
print(f"Registros sin dias de retraso y con pago por mora: {sin_dias_retra_con_pago}")

Registros con dias de retraso y sin pago por mora: 0
Registros sin dias de retraso y con pago por mora: 1


In [70]:
df_pagos_tra[(df_pagos_tra.dias_retraso == 0) & (df_pagos_tra.monto_mora_pagado != 0)]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
130171,130172,6245,5,2023-10-20,2023-11-20,4518.32,219.89,4298.43,4588.35,219.89,4298.43,70.03,478270.09,0,Muy Tardío,APP Móvil,REF0000130172


Lo que pasa para este registro es que calculando de manera manual, el dia de retraso sale 0.49091, al redondear se va directamente a 0, pero igualmente tiene monto_mora, entonces, considero que seria mejor reemplazar este dia de retraso por 1.  
**Nota**: Como solo es 1 registro no considero que sea necesario realizar un script complejo en lugar ello simplemente corrijo de manera manual.

In [71]:
df_pagos_tra.loc[df_pagos_tra.pago_id == 130172,'dias_retraso'] = 1 

In [72]:
dias_retra_sin_pago = df_pagos_tra[(df_pagos_tra.dias_retraso != 0) & (df_pagos_tra.monto_mora_pagado == 0)].shape[0]
sin_dias_retra_con_pago = df_pagos_tra[(df_pagos_tra.dias_retraso == 0) & (df_pagos_tra.monto_mora_pagado != 0)].shape[0]
print(f"Registros con dias de retraso y sin pago por mora: {dias_retra_sin_pago}")
print(f"Registros sin dias de retraso y con pago por mora: {sin_dias_retra_con_pago}")

Registros con dias de retraso y sin pago por mora: 0
Registros sin dias de retraso y con pago por mora: 0


In [73]:
#============================================================================================
# Verifica si los dias de retraso del ultimo pago coinciden con los dias de mora de prestamos 
#============================================================================================
# Seleccionamos algunas columnas a utilizar 
df_veri_dias_retraso = df_pagos_tra[['pago_id','prestamo_id','dias_retraso','numero_cuota','fecha_pago']].copy()
df_prestamos_veri_dias_retraso = df_prestamos_tra[['prestamo_id','numero_cuotas_total','dias_mora']].copy()

# Obtenemos el indice de la ultima cuota  
df_veri_dias_retraso = df_veri_dias_retraso.groupby('prestamo_id').agg(
    idx_cuota_maxima = ('numero_cuota','idxmax'),
).reset_index()
# obtemos los valores asociados a estos indices 
df_veri_dias_retraso = df_pagos_tra.loc[df_veri_dias_retraso.idx_cuota_maxima,['pago_id','prestamo_id','dias_retraso','numero_cuota','fecha_pago']].copy()
# LEFT JOIN, prestamo_id
df_merge_veri_dias_retraso = df_veri_dias_retraso.merge(
    right = df_prestamos_veri_dias_retraso,
    on='prestamo_id',
    how='left'
)

# Muestra los registros donde los dias de retraso no coinciden con los dias de mora, del ultimo pago 
# Resultados Esperados: Tabla Vacia
# Esta limpieza se va a desarrolar en 09_validacion_pagos_prestamos
df_revisar_dias_mora = df_merge_veri_dias_retraso[df_merge_veri_dias_retraso.dias_retraso != df_merge_veri_dias_retraso.dias_mora].copy()
df_revisar_dias_mora

,pago_id,prestamo_id,dias_retraso,numero_cuota,fecha_pago,numero_cuotas_total,dias_mora
3,86,4,16,6,2024-11-21,6,0
8,238,9,93,48,2024-06-22,48,0
9,281,10,0,43,2024-12-28,54,252
12,345,13,0,12,2024-04-25,12,52
13,363,14,73,18,2023-11-17,18,0
...,...,...,...,...,...,...,...
6485,134757,6486,47,18,2023-07-27,18,0
6486,134766,6487,30,9,2023-01-11,9,0
6489,134832,6490,0,30,2024-01-11,30,340
6490,134856,6491,24,24,2023-03-05,24,0


In [74]:
df_revisar_dias_mora["ERROR"] = "DIAS DE RETRASO NO COINCIDE CON DIAS DE MORA"

## fecha_pago V2

In [75]:
df_pagos_tra['fecha_pago'] = df_pagos_tra.apply(
    axis = 1,
    func = lambda x: 
        x['fecha_vencimiento_cuota'] + pd.DateOffset(days=x['dias_retraso'])
)
# Muestra los registros donde los dias de retraso, no sean iguales a la diferencia en dias de fecha pago y fecha vencimiento cuota 
df_pagos_tra[df_pagos_tra.dias_retraso != (df_pagos_tra.fecha_pago - df_pagos_tra.fecha_vencimiento_cuota).dt.days ]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [76]:
# Muestra las fechas de pago que son futura
df_pagos_tra[df_pagos_tra.fecha_pago.dt.date >= date.today()]

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago


In [77]:
#--------------------------------------------------------------------------------------------
# Verifica que la ultima fecha_pago de pagos, sea igual a fecha_ultimo_pago_real de prestamos
#--------------------------------------------------------------------------------------------
df_verificar_fechas_pag = df_pagos_tra[['pago_id','prestamo_id','numero_cuota','fecha_pago']].copy()
df_prestamos_veri_fechas = df_prestamos_tra[['prestamo_id','numero_cuotas_total','numero_cuotas_pagadas','numero_cuotas_pendientes','fecha_ultimo_pago_real']].copy()

# Obtenemos la ultima cuota y la ultima fecha de pago 
df_indice_ultima_cuota = df_verificar_fechas_pag.groupby('prestamo_id').agg(
    idx_max = ('numero_cuota','idxmax')
).reset_index()
df_verificar_fechas_pag = df_verificar_fechas_pag.loc[df_indice_ultima_cuota.idx_max]
# LEFT JOIN, prestamo_id
df_merge_veri_fechas = df_verificar_fechas_pag.merge(
    right = df_prestamos_veri_fechas,
    on='prestamo_id',
    how='left'
)
# Muestras los registros con fechas donde el ultimo pago no coincide 
# Resultados Esperados: Tabla Vacia
# Esta limpieza se va a desarrolar en 09_validacion_pagos_prestamos
df_revisar_fecha_ultimo_pago = df_merge_veri_fechas[df_merge_veri_fechas.fecha_pago != df_merge_veri_fechas.fecha_ultimo_pago_real].copy()
df_revisar_fecha_ultimo_pago

,pago_id,prestamo_id,numero_cuota,fecha_pago,numero_cuotas_total,numero_cuotas_pagadas,numero_cuotas_pendientes,fecha_ultimo_pago_real
3,86,4,6,2024-11-01,6,6,0,2024-10-16
8,238,9,48,2024-05-02,48,48,0,2024-01-30
13,363,14,18,2023-09-07,18,18,0,2023-06-26
17,469,18,38,2024-12-13,48,38,10,2024-12-07
24,578,25,4,2024-12-22,30,4,26,2024-12-21
...,...,...,...,...,...,...,...,...
6479,134677,6480,9,2024-08-03,9,9,0,2024-06-28
6485,134757,6486,18,2023-06-15,18,18,0,2023-04-29
6486,134766,6487,9,2022-12-04,9,9,0,2022-11-04
6490,134856,6491,24,2023-02-22,24,24,0,2023-01-29


In [78]:
df_revisar_fecha_ultimo_pago["ERROR"] = "FECHA ULTIMO PAGO REALIZADO NO COINCIDE CON LA FECHA ULTIMO PAGO REAL"

## estado_pago V2

**Puntual**: 0  
**Con Gracia**: 1 - 8  
**Tardío**: 9 - 30  
**Muy Tardío**: 31 - 150  

In [79]:
# Arreglamos el estado_pago segun la logica de negocio
df_pagos_tra['estado_pago'] = df_pagos_tra.dias_retraso.apply(formato_estado_pago)
df_pagos_tra.estado_pago.unique()

array(['Puntual', 'Con Gracia', 'Tardío', 'Muy Tardío'], dtype=object)

# Exportando la Tabla Limpia

In [80]:
df_pagos_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134911 entries, 0 to 134910
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   pago_id                     134911 non-null  int64         
 1   prestamo_id                 134911 non-null  int64         
 2   numero_cuota                134911 non-null  int64         
 3   fecha_vencimiento_cuota     134911 non-null  datetime64[ns]
 4   fecha_pago                  134911 non-null  datetime64[ns]
 5   monto_cuota_programada      134911 non-null  float64       
 6   monto_capital_programado    134911 non-null  float64       
 7   monto_interes_programado    134911 non-null  float64       
 8   monto_pagado_total          134911 non-null  float64       
 9   monto_capital_pagado        134911 non-null  float64       
 10  monto_interes_pagado        134911 non-null  float64       
 11  monto_mora_pagado           134911 non-

In [81]:
df_pagos_tra.head()

,pago_id,prestamo_id,numero_cuota,fecha_vencimiento_cuota,fecha_pago,monto_cuota_programada,monto_capital_programado,monto_interes_programado,monto_pagado_total,monto_capital_pagado,monto_interes_pagado,monto_mora_pagado,saldo_capital_despues_pago,dias_retraso,estado_pago,canal_pago,referencia_pago
0,1,1,1,2023-06-06,2023-06-06,393.63,238.91,154.72,393.63,238.91,154.72,0.0,8979.82,0,Puntual,APP Móvil,REF0000000001
1,2,1,2,2023-07-06,2023-07-06,393.63,242.92,150.71,393.63,242.92,150.71,0.0,8736.90,0,Puntual,Débito Automático,REF0000000002
2,3,1,3,2023-08-05,2023-08-05,393.63,247.00,146.63,393.63,247.00,146.63,0.0,8489.90,0,Puntual,Agencia,REF0000000003
3,4,1,4,2023-09-04,2023-09-04,393.63,251.14,142.49,393.63,251.14,142.49,0.0,8238.76,0,Puntual,Agencia,REF0000000004
4,5,1,5,2023-10-04,2023-10-04,393.63,255.36,138.27,393.63,255.36,138.27,0.0,7983.40,0,Puntual,Débito Automático,REF0000000005


In [82]:
df_pagos_tra.to_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_pagos.parquet"),
    index=False
)

# Exportando los Registros de Pagos a Revisar

In [83]:
df_para_verificar = pd.concat([df_revisar_dias_mora,df_revisar_fecha_ultimo_pago])
df_para_verificar.to_parquet(
    obtener_ruta_archivo("archivos_para_revision","revision_pagos.parquet"),
    index=False
)